# Actividad: poda alfa-beta en el gato

En clase resolvimos el gato con minimax y memoria. Ahora vas a agregarle **poda alfa-beta** y a medir cuánto trabajo ahorra.

**Cómo trabajar:** ejecuta las celdas en orden con **Mayúsculas + Enter**. Solo tienes que escribir código en la celda de la parte 1, llenar la tabla de la parte 3 y responder las preguntas de la parte 4.

**Material de apoyo** (en el repositorio https://github.com/ferdiex/gato):
- `minimax/minimax_alphabeta.py`: alfa-beta en el árbol de las láminas; tu código se parece mucho.
- `presentaciones/2_minimax_alfabeta_python.pdf`: la explicación paso a paso.
- La lámina 27 de la presentación M6 del curso: el pseudocódigo de MAX-VALUE.

**Entrega:** en Colab, menú *Archivo → Descargar → Descargar .ipynb*, y sube ese archivo a Eminus.

In [ ]:
# Preparación: descarga gato.py y define las herramientas de revisión.
# No necesitas modificar esta celda.
import os, math, time, urllib.request
if not os.path.exists("gato.py"):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/ferdiex/gato/main/gato.py", "gato.py")
import gato

llamadas = 0
REFERENCIA = {False: 18297, True: 7275}   # llamadas de la versión de las láminas

def todos_los_tableros():
    vistos, pila = set(), ["-" * 9]
    while pila:
        b = pila.pop()
        if b in vistos:
            continue
        vistos.add(b)
        if not gato.terminal(b):
            p = gato.to_move(b)
            pila.extend(gato.play(b, i, p) for i in gato.moves(b))
    return sorted(b for b in vistos if not gato.terminal(b))

def partidas(board, agente, f):
    if gato.terminal(board):
        u = gato.utility(board)
        return 1, int((u == 1 and agente == "o") or (u == -1 and agente == "x"))
    p = gato.to_move(board)
    if p == agente:
        _, i = f(board)
        if i not in gato.moves(board):
            raise ValueError(f"en el tablero {board} propone la jugada {i}, que no es legal")
        return partidas(gato.play(board, i, p), agente, f)
    t = d = 0
    for i in gato.moves(board):
        a, b = partidas(gato.play(board, i, p), agente, f)
        t, d = t + a, d + b
    return t, d

def contar(f, board="-" * 9):
    global llamadas
    llamadas = 0
    f(board)
    return llamadas

def verificar(f):
    try:
        v, j = f("-" * 9)
    except NotImplementedError as e:
        print("✗", e); return False
    except Exception as e:
        print(f"✗ Tu función produjo un error: {type(e).__name__}: {e}"); return False
    ok = True
    if v == 0:
        print("✓ El valor del tablero vacío es 0 (empate)")
    else:
        print(f"✗ El valor del tablero vacío debería ser 0 y tu función da {v}"); ok = False
    if j in gato.moves("-" * 9):
        print(f"✓ Propone una jugada legal para X: casilla {j}")
    else:
        print(f"✗ La jugada propuesta para el tablero vacío no es válida: {j}"); ok = False
    tableros = todos_los_tableros()
    try:
        malos = [b for b in tableros if f(b)[0] != gato.minimax(b)[0]]
    except Exception as e:
        print(f"✗ Tu función produjo un error en algún tablero: {type(e).__name__}: {e}"); return False
    if not malos:
        print(f"✓ Da el mismo valor que minimax en los {len(tableros):,} tableros posibles")
    else:
        ok = False
        b = malos[0]
        print(f"✗ En {len(malos):,} de {len(tableros):,} tableros el valor no coincide con minimax. Por ejemplo:")
        gato.show(b)
        print(f"  minimax: {gato.minimax(b)[0]}   tu función: {f(b)[0]}")
    for ag in ("x", "o"):
        try:
            t, d = partidas("-" * 9, ag, f)
        except Exception as e:
            print(f"✗ Jugando con {ag.upper()}: {e}"); ok = False; continue
        if d == 0:
            print(f"✓ Jugando con {ag.upper()}: {t} partidas posibles, 0 derrotas")
        else:
            print(f"✗ Jugando con {ag.upper()}: pierde {d} de {t} partidas"); ok = False
    if ok:
        n, ref = contar(f), REFERENCIA[bool(globals().get("ORDEN_CENTRO", False))]
        if n >= 549946:
            print(f"✗ Hace {n:,} llamadas, igual que minimax sin poda: no está podando."); ok = False
        elif n > ref:
            print(f"⚠ Correcto, pero hace {n:,} llamadas y la versión de las láminas hace {ref:,}.")
            print("  Revisa las condiciones de poda (≥ y ≤, como en la lámina 27 del curso, M6) y qué cota actualiza cada función.")
        else:
            print(f"✓ Hace {n:,} llamadas para resolver el tablero vacío")
    print("\nTodo correcto." if ok else "\nRevisa los puntos marcados con ✗.")
    return ok

print("Listo: gato.py cargado")

## 1. Tu alfa-beta

Completa la función con poda alfa-beta. Usa las mismas reglas de `gato.py`, igual que en clase:

| Función | Qué hace |
|---|---|
| `gato.moves(board)` | lista de casillas libres |
| `gato.play(board, i, player)` | tablero nuevo con la marca de `player` en la casilla `i` |
| `gato.to_move(board)` | `"x"` u `"o"`: a quién le toca |
| `gato.terminal(board)` | `True` si la partida terminó |
| `gato.utility(board)` | `1` si gana X, `-1` si gana O, `0` si empatan |

X es MAX y O es MIN, como en las láminas. La función regresa una pareja `(valor, mejor_jugada)`.

Llena los huecos numerados del 1 al 4.

In [ ]:
import math

ORDEN_CENTRO = False                       # parte 3: después cámbialo a True
PREFERENCIA = [4, 0, 2, 6, 8, 1, 3, 5, 7]  # centro, esquinas, orillas

def orden(movs):
    if ORDEN_CENTRO:
        return sorted(movs, key=PREFERENCIA.index)
    return movs


def alphabeta(board, alpha=-math.inf, beta=math.inf):
    """Return (value, best_move) using alpha-beta pruning."""
    global llamadas
    llamadas += 1                          # no borres esta línea: cuenta las llamadas
    if gato.terminal(board):
        return gato.utility(board), None
    if gato.to_move(board) == "x":
        return max_value(board, alpha, beta)
    return min_value(board, alpha, beta)


def max_value(board, alpha, beta):
    v, best = -math.inf, None
    for i in orden(gato.moves(board)):
        child_v, _ = alphabeta(gato.play(board, i, "x"), alpha, beta)
        # 1. Si child_v es mayor que v, guarda child_v en v y la casilla i en best.

        # 2. Poda beta: si v >= beta, regresa (v, best) sin revisar las demás casillas.

        # 3. Actualiza alpha con el mayor entre alpha y v.

    return v, best


def min_value(board, alpha, beta):
    # 4. Escribe la versión espejo de max_value:
    #    v empieza en +infinito, O coloca "o", se queda con el valor menor,
    #    poda si v <= alpha y actualiza beta con el menor entre beta y v.
    raise NotImplementedError("Completa min_value: es la versión espejo de max_value.")

## 2. Verifica tu función

Esta celda revisa tu alfa-beta en **todos** los tableros posibles del gato: compara su valor con el de minimax y juega todas las partidas posibles contra él. Si algo falla, te dice qué y te muestra un ejemplo.

Repite: corrige la celda de la parte 1, ejecútala de nuevo y vuelve a verificar, hasta ver *Todo correcto*.

> Alfa-beta garantiza el mismo **valor** que minimax, pero no necesariamente la misma **jugada** cuando hay empates. Por eso la revisión compara valores y no jugadas.

In [ ]:
_ = verificar(alphabeta)

## 3. Mide y compara

**Antes de ejecutar la siguiente celda**, escribe tu predicción de cuántas llamadas hace cada versión para resolver el tablero vacío. Haz doble clic en esta celda para editarla.

| Versión | Tu predicción | Medido |
|---|---|---|
| Minimax sin memoria | | |
| Minimax con memoria | | |
| Tu alfa-beta, orden natural | | |
| Tu alfa-beta, centro primero | | |

Después de ejecutarla, copia los valores medidos en la última columna.

In [ ]:
# Mide las cuatro versiones al resolver el tablero vacío.
# minimax sin memoria se reescribe aquí solo para contar sus llamadas.
def minimax_sin_memoria(board):
    global llamadas
    llamadas += 1
    if gato.terminal(board):
        return gato.utility(board), None
    p = gato.to_move(board)
    pares = [(minimax_sin_memoria(gato.play(board, i, p))[0], i) for i in gato.moves(board)]
    return max(pares) if p == "x" else min(pares)

gato.minimax.cache_clear()
gato.minimax("-" * 9)
filas = [("Minimax sin memoria", contar(minimax_sin_memoria)),
         ("Minimax con memoria (@lru_cache)", gato.minimax.cache_info().misses)]
original = ORDEN_CENTRO
try:
    for centro, nombre in ((False, "Tu alfa-beta, orden natural"), (True, "Tu alfa-beta, centro primero")):
        ORDEN_CENTRO = centro
        filas.append((nombre, contar(alphabeta)))
except NotImplementedError as e:
    print("Primero completa tu alfa-beta:", e)
finally:
    ORDEN_CENTRO = original

for nombre, n in filas:
    print(f"{nombre:<36}{n:>10,} llamadas")

## 4. Preguntas

Responde en la celda de texto al final del cuaderno.

**Pregunta 1.** En `minimax`, `@lru_cache` guarda un resultado por cada tablero. ¿Se puede hacer lo mismo en alfa-beta, guardando un resultado por tablero? La celda siguiente hace el experimento. Ejecútala y explica qué pasa y por qué.

**Pregunta 2.** En la poda usamos `v >= beta` y `v <= alpha`, como en la lámina 27 del curso (M6). ¿Qué crees que pasaría con el número de llamadas si usaras `>` y `<`? Pruébalo y explica el resultado. *Pista:* en el gato, ¿cuántos valores distintos puede tener una partida?

**Pregunta 3.** ¿Qué tipo de trabajo evita la memoria y qué tipo de trabajo evita alfa-beta? Usa tu tabla de la parte 3.

In [ ]:
# Experimento opcional: alfa-beta con una memoria "ingenua" que guarda
# el resultado de cada tablero sin tomar en cuenta alpha y beta.
memoria = {}
alphabeta_original = alphabeta

def alphabeta_con_memoria(board, alpha=-math.inf, beta=math.inf):
    if board not in memoria:
        memoria[board] = alphabeta_original(board, alpha, beta)
    return memoria[board]

alphabeta = alphabeta_con_memoria        # max_value y min_value ahora llaman a esta
_ = verificar(alphabeta)
alphabeta = alphabeta_original           # restauramos tu versión

### Mis respuestas

**Pregunta 1:**

**Pregunta 2:**

**Pregunta 3:**

## Opcional: en Prolog

Si quieres un reto extra, agrega alfa-beta a `gato.pl` siguiendo `minimax/minimax_alphabeta.pl` del repositorio. La diferencia con Python es que el ciclo `for` se convierte en recursión sobre la lista de casillas libres.